In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
def compute_load_balancing_loss(
    routing_weights: torch.Tensor, 
    selected_experts: torch.Tensor, 
    num_experts: int, 
    top_k: int,
    alpha: float = 0.01
):
    """
    计算 MoE 的负载均衡辅助损失（支持 Top-K 路由）
    
    Args:
        routing_weights: [batch_size * seq_len, top_k]，每个 token 选中的 K 个专家的权重（已归一化）
        selected_experts: [batch_size * seq_len, top_k]，每个 token 选中的 K 个专家的索引
        num_experts: 专家总数 E
        top_k: 每个 token 选择的专家数量 K
        alpha: 损失权重系数
    
    Returns:
        aux_loss: 标量，负载均衡损失
    """
    batch_size_x_seq_len, _ = selected_experts.shape
    total_tokens = batch_size_x_seq_len
    
    # ==========================================
    # TODO 1: 计算 P_i（每个专家的平均路由概率得分）
    # ==========================================
    # P_i = ???
    
    # ==========================================
    # TODO 2: 计算 f_i（每个专家实际分到的 Token 比例）
    # ==========================================
    # expert_mask = ???
    # tokens_per_expert = ???
    # f_i = ???
    
    # ==========================================
    # TODO 3: 计算最终的 auxiliary loss
    # ==========================================
    # aux_loss = ???

    aux_loss = torch.tensor(0.0, device=routing_weights.device)
                                                                                                                                                                                  
    return aux_loss

In [ ]:
# 测试你的实现
def test_aux_loss():
    try:
        torch.manual_seed(42)
        num_experts = 8
        top_k = 2
        num_tokens = 1000
        alpha = 0.01
        
        # 模拟路由结果
        # 1. 极度不均衡：所有 token 都选专家 0 和 1
        bad_selected = torch.zeros(num_tokens, top_k, dtype=torch.long)
        bad_selected[:, 0] = 0
        bad_selected[:, 1] = 1
        bad_weights = torch.ones(num_tokens, top_k) / top_k
        
        loss_bad = compute_load_balancing_loss(bad_weights, bad_selected, num_experts, top_k, alpha)
        
        # 2. 绝对均匀：token 均匀分配给所有专家
        good_selected = torch.zeros(num_tokens, top_k, dtype=torch.long)
        for i in range(num_tokens):
            good_selected[i, 0] = (i * 2) % num_experts
            good_selected[i, 1] = (i * 2 + 1) % num_experts
        good_weights = torch.ones(num_tokens, top_k) / top_k
        
        loss_good = compute_load_balancing_loss(good_weights, good_selected, num_experts, top_k, alpha)
        
        print(f"极度不均衡的 Loss: {loss_bad.item():.4f}")
        print(f"绝对均匀的 Loss  : {loss_good.item():.4f}")
        
        # 理论最小值：当完全均匀时，P_i = 1/(E*top_k), f_i = 1/E
        # sum(f_i * P_i) = E * (1/E) * (1/(E*top_k)) = 1/(E*top_k)
        # aux_loss = alpha * E * 1/(E*top_k) = alpha / top_k
        expected_min = alpha / top_k  # 0.01 / 2 = 0.005
        assert torch.allclose(loss_good, torch.tensor(expected_min), atol=1e-4), f"理论最小 Loss 计算错误！期望 {expected_min:.4f}，实际 {loss_good.item():.4f}"
        assert loss_bad > loss_good * 2, "惩罚项没有对不均衡分布产生足够大的 Loss！"
        
        print("\n✅ All Tests Passed! 你成功掌握了 Mixtral / DeepSeek 的防崩塌核心技术！")
        
    except NotImplementedError:
        print("请先完成 TODO 代码！")
    except TypeError:
        print("代码未完成导致返回 None 错误。")
    except Exception as e:
        print(f"❌ 测试失败: {e}")
        raise e
    except Exception as e:
        print(f"\n❌ 发生未知异常: {type(e).__name__}: {e}")
        raise e

test_aux_loss()

In [ ]:
# --- Trick 1: Gemma 风格的 RMSNorm ---
class GemmaRMSNorm(nn.Module):
    def __init__(self, hidden_size: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        # weight 初始化为全 0
        self.weight = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 计算均方根
        x_f32 = x.float()
        variance = x_f32.pow(2).mean(-1, keepdim=True)
        x_norm = x_f32 * torch.rsqrt(variance + self.eps)
        
        # ==========================================
        # TODO 1: 实现 Gemma 的 +1 缩放
        # 注意类型转换回 x.dtype
        # ==========================================
        # output = ???

        # 占位初始化（返回错误值，确保数值测试失败）                                                                                                                                 
        output = torch.zeros_like(x)                                                                                                                                                 
  
        return output     
        



# --- Trick 2: Qwen 风格的权重绑定 ---
class QwenTieEmbeddings(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int):
        super().__init__()
        # 1. 定义标准的 Embedding 层
        self.embed_tokens = nn.Embedding(vocab_size, hidden_size)
        
        # 2. 定义最后的 LM Head 预测层，注意不要 bias
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)
        
        # ==========================================
        # TODO 2: 将 lm_head 的权重在内存级别绑定到 embed_tokens 上
        # 提示: 在 PyTorch 中，可以直接赋值 nn.Parameter 或是底层 tensor
        # self.lm_head.weight = ???
        # ==========================================
        # ???
        
    def forward_embed(self, input_ids):
        return self.embed_tokens(input_ids)
        
    def forward_lm_head(self, hidden_states):
        return self.lm_head(hidden_states)

In [ ]:
# 测试你的实现
def test_tricks():
    try:
        hidden_size = 64
        vocab_size = 1000
        
        # 1. 测试 Gemma RMSNorm
        gemma_norm = GemmaRMSNorm(hidden_size)
        x = torch.randn(2, 10, hidden_size)
        out = gemma_norm(x)
        
        # 验证初始化时 (weight=0)，输出等价于无缩放的 norm
        variance = x.float().pow(2).mean(-1, keepdim=True)
        expected = (x.float() * torch.rsqrt(variance + 1e-6)).to(x.dtype)
        
        assert torch.allclose(out, expected, atol=1e-4), "Gemma 的 1+w 缩放机制实现错误！"
        print("✅ Gemma RMSNorm (+1 trick) 测试通过！")
        
        # 2. 测试 Qwen 权重绑定
        qwen_model = QwenTieEmbeddings(vocab_size, hidden_size)
        
        # 检查物理内存地址是否相同
        ptr_embed = qwen_model.embed_tokens.weight.data_ptr()
        ptr_head = qwen_model.lm_head.weight.data_ptr()
        assert ptr_embed == ptr_head, "权重未在物理内存级别绑定！"
        
        # 模拟训练更新一次 Embedding
        qwen_model.embed_tokens.weight.data += 1.0
        
        # 验证 LM Head 的权重也跟着变了 (因为它们是同一个指针)
        assert qwen_model.lm_head.weight.data[0, 0] == qwen_model.embed_tokens.weight.data[0, 0], "权重更新未同步！"
        
        print("✅ Qwen Tie Word Embeddings 权重绑定测试通过！")
        print("\n架构变体技巧测试通过。")
        
    except NotImplementedError:
        print("请先完成 TODO 代码！")
    except AttributeError:
        print("代码未完成导致变量属性错误。")
    except AssertionError as e:
        print(f"❌ 测试失败: {e}")
        raise e
    except Exception as e:
        print(f"❌ 发生未知异常: {e}")
        raise e

test_tricks()